# Phase 1 — Member B: B4, B5, B6 (Real Data Fine-Tuning + Retrieval API)

Runs the real-data continuation of Member B's work:
- **B4** — build real training pairs/triplets from Member A's Sync Point 1 handoff
- **B5** — fine-tune on real data, benchmark against QRCD
- **B6** — build the vector index and retrieval API for Phase 2

**Recommended:** Runtime → Change runtime type → GPU (speeds up B5 fine-tuning).

**Before running:** you need TWO things uploaded:
1. `MemberB_B4_B5_B6_Code.zip` (this code)
2. Member A's `final_cross_reference_index.csv` (the real Sync Point 1 data) — either as its own file, or inside your full repo zip


## 1. Upload the code zip

In [ ]:
from google.colab import files

print("Select MemberB_B4_B5_B6_Code.zip")
uploaded_code = files.upload()


Select MemberB_B4_B5_B6_Code.zip


Saving MemberB_B4_B5_B6_Code.zip to MemberB_B4_B5_B6_Code.zip


In [ ]:
!unzip -o -q MemberB_B4_B5_B6_Code.zip -d MemberB
%cd MemberB
!find . -type f


/content/MemberB
./requirements.txt
./src/b3_test_harness.py
./src/b6_build_index_and_retrieval_api.py
./src/b1_load_model.py
./src/b2_finetune_harness.py
./src/b4_build_real_triplets.py
./src/b5_finetune_and_benchmark.py
./src/results_utils.py
./README.md


## 2. Upload Member A's real data

Choose ONE of these two options depending on what you have:


### Option B — you just have the single CSV file

In [ ]:
from google.colab import files
import os

print("Select final_cross_reference_index.csv directly")
uploaded_csv = files.upload()

os.makedirs("quranNLP/shared/data", exist_ok=True)
csv_name = list(uploaded_csv.keys())[0]
import shutil
shutil.move(csv_name, "quranNLP/shared/data/final_cross_reference_index.csv")
print("Placed at quranNLP/shared/data/final_cross_reference_index.csv")


Select final_cross_reference_index.csv directly


Saving final_cross_reference_index.csv to final_cross_reference_index.csv
Placed at quranNLP/shared/data/final_cross_reference_index.csv


## 3. Confirm the data is in place

In [ ]:
import pandas as pd

path = "quranNLP/shared/data/final_cross_reference_index.csv"
if os.path.exists(path):
    df = pd.read_csv(path)
    print(f"Loaded {len(df)} rows.")
    display(df.head())
else:
    print("Data file not found - go back and run one of the upload options above.")


Loaded 6236 rows.


,verse_key,clean_verse,related_verse_keys,tafsir_passage,has_direct_tafsir
0,1:1,بسم الله الرحمـٰن الرحيم,1:2,بِسْمِ اللَّهِ الرَّحْمَنِ الرَّحِيمِ\n\nفَاتِ...,True
1,1:2,الحمد لله رب العـٰلمين,1:3,الْقُرَّاءُ السَّبْعَةُ عَلَى ضَمِّ الدَّالِ م...,True
2,1:3,الرحمـٰن الرحيم,1:4,وَقَوْلُهُ: ﴿الرَّحْمَنِ الرَّحِيمِ﴾ تَقَدَّمَ...,True
3,1:4,مـٰلك يوم الدين,1:5,قَرَأَ بَعْضُ الْقُرَّاءِ: ﴿مَلِك يَوْمِ الدِّ...,True
4,1:5,اياك نعبد واياك نستعين,1:6,[قَرَأَ السَّبْعَةُ وَالْجُمْهُورُ بِتَشْدِيدِ...,True


## 4. Install dependencies

In [ ]:
!pip install -q -r requirements.txt


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## 5. B4 — Build real training pairs/triplets

In [ ]:
!python src/b4_build_real_triplets.py


[OK] Loaded 6236 rows from quranNLP/shared/data/final_cross_reference_index.csv
[OK] Built 6236 verse-tafsir pairs (from 6236/6236 rows with direct tafsir).
[OK] Built 6122 verse-to-related-verse pairs.
[OK] Built 12358 triplets (anchor, positive, cross-surah negative).

[PASS] Negative/positive collision check (0 collisions out of 12358 triplets).
[SAVED] data/real_training_pairs.json (12358 pairs)
[SAVED] data/real_training_triplets.json (12358 triplets)

[RESULT] B4 real triplet construction PASSED.


## 6. B5 — Fine-tune and benchmark
Fast offline check first, then the real run.

In [ ]:
!python src/b5_finetune_and_benchmark.py --dry-run


[OK] Loaded 12358 real pairs, 12358 real triplets.
[00:00:00] Tokenize words                 ██████████████████ 173258   /   173258
[00:00:00] Count pairs                    ██████████████████ 173258   /   173258
[00:00:00] Compute merges                 ██████████████████ 694      /      694
Writing model shards: 100% 1/1 [00:00<00:00, 460.71it/s]
Loading weights: 100% 39/39 [00:00<00:00, 6640.60it/s]
/content/MemberB/src/b3_test_harness.py:141: FutureWarning: The `get_word_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  pooling_model = models.Pooling(word_embedding_model.get_word_embedding_dimension())
Writing model shards: 100% 1/1 [00:00<00:00, 302.82it/s]
[OK] Built tiny dry-run base model at: ./_tiny_dry_run_base (vocab=800, hidden=32, layers=2) - no network used.

[STEP] Fine-tuning on 12358 real pairs / 12358 real triplets (dry_run=True)...
Loading weights: 100% 39/39 [00:00<00:00, 6322.83it/s]
/content/MemberB/src/b2_finetune_harness.py:57: FutureW

In [ ]:
from google.colab import files
import shutil

print("Select b2_finetune_harness.py and b5_finetune_and_benchmark.py together")
uploaded = files.upload()

for filename in uploaded.keys():
    dest = f"src/{filename}"
    shutil.move(filename, dest)
    print(f"Replaced: {dest}")

Select b2_finetune_harness.py and b5_finetune_and_benchmark.py together


Saving b5_finetune_and_benchmark.py to b5_finetune_and_benchmark.py
Replaced: src/b5_finetune_and_benchmark.py


In [ ]:
!ls -la src/b2_finetune_harness.py src/b5_finetune_and_benchmark.py
!grep "per_device_train_batch_size: int = 32" src/b2_finetune_harness.py && echo "b2 OK - new version confirmed"
!grep "default=64" src/b5_finetune_and_benchmark.py && echo "b5 OK - new version confirmed"

-rw-r--r-- 1 root root 8264 Jul 13 04:14 src/b2_finetune_harness.py
-rw-r--r-- 1 root root 7424 Jul 13 04:14 src/b5_finetune_and_benchmark.py
    per_device_train_batch_size: int = 32
b2 OK - new version confirmed
    parser.add_argument("--batch-size", type=int, default=64,
    parser.add_argument("--max-seq-length", type=int, default=64,
b5 OK - new version confirmed


In [ ]:
!rm -rf b5_real_finetuned
!python src/b5_finetune_and_benchmark.py

[OK] Loaded 12358 real pairs, 12358 real triplets.
[CONFIG] epochs=2, batch_size=64, max_seq_length=64, matryoshka_dims=[768, 256, 64], fp16=True

[STEP] Fine-tuning on 12358 real pairs / 12358 real triplets (dry_run=False)...
Loading weights: 100% 199/199 [00:00<00:00, 5370.95it/s]
[OK] Capped max_seq_length to 64 (shorter sequences = faster training, and Quranic verses/tafsir sentences rarely need the model's full default length).
/content/MemberB/src/b2_finetune_harness.py:65: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  f"(native dim = {model.get_sentence_embedding_dimension()})")
[OK] Base model loaded: Omartificial-Intelligence-Space/GATE-AraBert-v1 (native dim = 768)
[OK] Built hybrid loss: contrastive (weight=1.0) + triplet (weight=1.0), Matryoshka dims=[768, 256, 64]
[OK] Built datasets: contrastive=12358 pairs, triplet=12358 triplets
The `warmup_ratio` argument is deprecated in Transformers v5+, and will also be

## 7. B6 — Build the vector index and retrieval API

In [ ]:
!ls b5_real_finetuned/config.json && echo "real model saved correctly"
!python src/b6_build_index_and_retrieval_api.py --model-path ./b5_real_finetuned

b5_real_finetuned/config.json
real model saved correctly
Loading weights: 100% 199/199 [00:00<00:00, 6476.31it/s]
[OK] Built 12472 retrievable entries (6236 verses, 6236 tafsir passages).
Batches: 100% 390/390 [01:04<00:00,  6.05it/s]
[OK] Built HNSW index: 12472 items, dim=768
[SAVED] Index and metadata written to index/

Retrieval verification:

  Query: الرحمة والمغفرة
    [verse] 1:3 (sim=0.849): الرحمـٰن الرحيم...
    [verse] 80:13 (sim=0.833): فى صحف مكرمة...
    [verse] 93:1 (sim=0.832): والضحىٰ...

  Query: الصبر على البلاء
    [verse] 53:38 (sim=0.772): الا تزر وازرة وزر اخرىٰ...
    [verse] 74:7 (sim=0.768): ولربك فاصبر...
    [verse] 74:47 (sim=0.764): حتىٰ اتىٰنا اليقين...

[PASS] Retrieval returned results with provenance metadata.

[RESULT] B6 vector index + retrieval API PASSED verification.
Ready for Phase 2's agentic retrieval loop to call RetrievalAPI.retrieve(query, top_k).


## 8. Try the retrieval API directly

In [ ]:
import sys
sys.path.insert(0, "src")
from b6_build_index_and_retrieval_api import load_index, RetrievalAPI
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("./b5_real_finetuned")
index, entries = load_index(dim=model.get_sentence_embedding_dimension())
api = RetrievalAPI(model, index, entries)

results = api.retrieve("الرحمة والمغفرة", top_k=5)
for r in results:
    print(f"[{r['source_type']}] {r['verse_key']} (sim={r['similarity']:.3f}): {r['text'][:80]}")


ValueError: Unrecognized model in ./b5_real_finetuned. Should have a `model_type` key in its config.json.

In [ ]:
!ls -la b5_real_finetuned
!find b5_real_finetuned -name "config.json"

total 530320
drwxr-xr-x  5 root root      4096 Jul 13 04:23 .
drwxr-xr-x 10 root root      4096 Jul 13 04:25 ..
drwxr-xr-x  2 root root      4096 Jul 13 04:23 1_Pooling
drwxr-xr-x  3 root root      4096 Jul 13 04:19 checkpoint-388
drwxr-xr-x  3 root root      4096 Jul 13 04:23 checkpoint-776
-rw-r--r--  1 root root       712 Jul 13 04:23 config.json
-rw-r--r--  1 root root       284 Jul 13 04:23 config_sentence_transformers.json
-rw-------  1 root root 540795752 Jul 13 04:23 model.safetensors
-rw-r--r--  1 root root       277 Jul 13 04:23 modules.json
-rw-r--r--  1 root root    425673 Jul 13 04:23 README.md
-rw-r--r--  1 root root       241 Jul 13 04:23 sentence_bert_config.json
-rw-r--r--  1 root root       710 Jul 13 04:23 tokenizer_config.json
-rw-r--r--  1 root root   1777114 Jul 13 04:23 tokenizer.json
b5_real_finetuned/checkpoint-388/1_Pooling/config.json
b5_real_finetuned/checkpoint-388/config.json
b5_real_finetuned/1_Pooling/config.json
b5_real_finetuned/checkpoint-776/1_Poolin

In [ ]:
!cat b5_real_finetuned/config.json

{
  "add_cross_attention": false,
  "architectures": [
    "BertModel"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "classifier_dropout": null,
  "dtype": "float32",
  "eos_token_id": null,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "tie_word_embeddings": true,
  "transformers_version": "5.12.1",
  "type_vocab_size": 2,
  "use_cache": false,
  "vocab_size": 64000
}


In [ ]:
import sys
sys.path.insert(0, "src")
from b6_build_index_and_retrieval_api import load_index, RetrievalAPI
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("./b5_real_finetuned")
print("Loaded successfully")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loaded successfully


In [ ]:
index, entries = load_index(dim=model.get_sentence_embedding_dimension())
api = RetrievalAPI(model, index, entries)

results = api.retrieve("الرحمة والمغفرة", top_k=5)
for r in results:
    print(f"[{r['source_type']}] {r['verse_key']} (sim={r['similarity']:.3f}): {r['text'][:80]}")

/tmp/ipykernel_521/1531431995.py:1: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  index, entries = load_index(dim=model.get_sentence_embedding_dimension())


[verse] 1:3 (sim=0.849): الرحمـٰن الرحيم
[verse] 80:13 (sim=0.833): فى صحف مكرمة
[verse] 55:1 (sim=0.832): الرحمـٰن
[verse] 93:1 (sim=0.832): والضحىٰ
[verse] 56:94 (sim=0.830): وتصلية جحيم


In [ ]:
!cat outputs/b5_benchmark_*.json

{
  "num_pairs": 12358,
  "num_triplets": 12358,
  "qrcd_top1_accuracy": null,
  "output_dir": "./b5_real_finetuned_dryrun"
}{
  "num_pairs": 12358,
  "num_triplets": 12358,
  "qrcd_top1_accuracy": null,
  "output_dir": "./b5_real_finetuned"
}

## 9. Save everything to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DEST = "/content/drive/MyDrive/Phase1_Project/MemberB_B4_B6_output"
!mkdir -p "{DEST}"
!cp -r data "{DEST}/" 2>/dev/null
!cp -r outputs "{DEST}/" 2>/dev/null
!cp -r b5_real_finetuned "{DEST}/" 2>/dev/null
!cp -r index "{DEST}/" 2>/dev/null

print(f"Saved to: {DEST}")
!find "{DEST}" -maxdepth 2


Mounted at /content/drive
Saved to: /content/drive/MyDrive/Phase1_Project/MemberB_B4_B6_output
/content/drive/MyDrive/Phase1_Project/MemberB_B4_B6_output
/content/drive/MyDrive/Phase1_Project/MemberB_B4_B6_output/data
/content/drive/MyDrive/Phase1_Project/MemberB_B4_B6_output/data/real_training_pairs.json
/content/drive/MyDrive/Phase1_Project/MemberB_B4_B6_output/data/real_training_triplets.json
/content/drive/MyDrive/Phase1_Project/MemberB_B4_B6_output/outputs
/content/drive/MyDrive/Phase1_Project/MemberB_B4_B6_output/outputs/b5_benchmark_20260713_034337.json
/content/drive/MyDrive/Phase1_Project/MemberB_B4_B6_output/outputs/b5_benchmark_20260713_042347.json
/content/drive/MyDrive/Phase1_Project/MemberB_B4_B6_output/b5_real_finetuned
/content/drive/MyDrive/Phase1_Project/MemberB_B4_B6_output/b5_real_finetuned/checkpoint-388
/content/drive/MyDrive/Phase1_Project/MemberB_B4_B6_output/b5_real_finetuned/checkpoint-776
/content/drive/MyDrive/Phase1_Project/MemberB_B4_B6_output/b5_real_fine

In [ ]:
from google.colab import files
uploaded = files.upload()  # select generate_phase1_graphs.py
import shutil
shutil.move("generate_phase1_graphs.py", "src/generate_phase1_graphs.py")

!pip install -q matplotlib
!python src/generate_phase1_graphs.py --model-path ./b5_real_finetuned --index-dir index

Saving generate_phase1_graphs.py to generate_phase1_graphs.py
[STEP] Generating Phase 1 result graphs from real saved artifacts...

[SAVED] graphs/02_dataset_sizes.png
[SAVED] graphs/01_training_loss.png (38 logged steps)
[SKIP] qrcd_top1_accuracy not present (QRCD data wasn't available during B5).
Loading weights: 100% 199/199 [00:00<00:00, 5556.96it/s]
/content/MemberB/src/generate_phase1_graphs.py:171: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  index, entries = load_index(dim=model.get_sentence_embedding_dimension(), out_dir=index_dir)
[SAVED] graphs/04_retrieval_quality.png

[RESULT] Generated 3/4 graphs in graphs/
  - graphs/02_dataset_sizes.png
  - graphs/01_training_loss.png
  - graphs/04_retrieval_quality.png

[NOTE] Some graphs were skipped because their source data wasn't found. Make sure B4, B5, and B6 have all been run before generating graphs.


In [ ]:
from google.colab import files
import os

print("Select BOTH qrcd_v1.1_train.json and qrcd_v1.1_test.json together")
uploaded = files.upload()

Select BOTH qrcd_v1.1_train.json and qrcd_v1.1_test.json together


Saving qrcd_v1.1_test.json to qrcd_v1.1_test.json
Saving qrcd_v1.1_train.json to qrcd_v1.1_train.json


In [ ]:
import json

records = []
for split_name, filename in [("train", "qrcd_v1.1_train.json"), ("test", "qrcd_v1.1_test.json")]:
    with open(filename, encoding="utf-8") as f:
        raw = json.load(f)

    for para in raw["data"]:
        passage = para["paragraphs"][0]["context"]
        for qa in para["paragraphs"][0]["qas"]:
            records.append({
                "split": split_name,
                "pq_id": qa["id"],
                "passage": passage,
                "question": qa["question"],
                "answers": [a["text"] for a in qa.get("answers", [])],
            })

os.makedirs("data", exist_ok=True)
with open("data/qrcd_flat.json", "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

print(f"Saved {len(records)} records to data/qrcd_flat.json")

Saved 1093 records to data/qrcd_flat.json


In [ ]:
!ls -la data/qrcd_flat.json

-rw-r--r-- 1 root root 1143729 Jul 13 04:49 data/qrcd_flat.json


In [ ]:
import sys
sys.path.insert(0, "src")
from sentence_transformers import SentenceTransformer
from b5_finetune_and_benchmark import evaluate_qrcd_retrieval
from results_utils import save_json_log

model = SentenceTransformer("./b5_real_finetuned")
accuracy = evaluate_qrcd_retrieval(model)
save_json_log("b5_benchmark", {"qrcd_top1_accuracy": accuracy})

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[OK] QRCD retrieval top-1 accuracy on 20 sample questions: 30.0%
[SAVED] JSON log: outputs/b5_benchmark_20260713_045010.json


'outputs/b5_benchmark_20260713_045010.json'

In [ ]:
!python src/generate_phase1_graphs.py --model-path ./b5_real_finetuned --index-dir index

[STEP] Generating Phase 1 result graphs from real saved artifacts...

[SAVED] graphs/02_dataset_sizes.png
[SAVED] graphs/01_training_loss.png (38 logged steps)
[SAVED] graphs/03_qrcd_benchmark.png
Loading weights: 100% 199/199 [00:00<00:00, 6158.31it/s]
/content/MemberB/src/generate_phase1_graphs.py:171: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  index, entries = load_index(dim=model.get_sentence_embedding_dimension(), out_dir=index_dir)
[SAVED] graphs/04_retrieval_quality.png

[RESULT] Generated 4/4 graphs in graphs/
  - graphs/02_dataset_sizes.png
  - graphs/01_training_loss.png
  - graphs/03_qrcd_benchmark.png
  - graphs/04_retrieval_quality.png


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DEST = "/content/drive/MyDrive/Phase1_Project/graphs"
!mkdir -p "{DEST}"
!cp -r graphs/* "{DEST}/"
print(f"Saved to: {DEST}")
!ls "{DEST}"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved to: /content/drive/MyDrive/Phase1_Project/graphs
01_training_loss.png  03_qrcd_benchmark.png
02_dataset_sizes.png  04_retrieval_quality.png
